# 3 - Record, train, and deploy in one loop

The full data loop in one notebook: **record** a LeRobotDataset, **train** an [ACT](https://tonyzhaozh.github.io/aloha/) policy on it, **export** the checkpoint, and **load** it back as a runnable policy. It runs on CPU - small dataset, two training steps - so you can see the whole loop close on a laptop with no GPU.

For a real policy you would raise the step count and run on a GPU, but the code path is identical. Part of the [examples/notebooks](./README.md) series.

In [ ]:
import os
# macOS uses the 'cgl' offscreen GL backend; Linux headless uses 'egl'.
os.environ.setdefault("MUJOCO_GL", "cgl")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")  # ACT loads through LeRobot
import shutil
ROOT, OUT = "/tmp/nb3_dataset", "/tmp/nb3_ft"
# LeRobot's trainer refuses to write into an existing output_dir, so clear it
# first to keep this notebook re-runnable.
shutil.rmtree(OUT, ignore_errors=True)

## 1. Record a small dataset

Same recording flow as [notebook 2](./02_record_and_stream.ipynb), kept short for a fast CPU run.

In [ ]:
from strands_robots import Robot, MockPolicy, create_policy
from strands_robots.training import TrainSpec, create_trainer

sim = Robot("so100", mesh=False)
sim.add_camera(name="front", position=[0.5, 0.0, 0.4], target=[0.2, 0.0, 0.05])
sim.start_recording(repo_id="local/nb3_demo", root=ROOT, fps=30,
                    task="pick up the red cube", overwrite=True)
sim.run_policy(robot_name="so100", policy_object=MockPolicy(),
               instruction="pick up the red cube", n_steps=60)
sim.stop_recording()
sim.destroy()
print("recorded ->", ROOT)

## 2. Train

`create_trainer(provider)` returns a `Trainer` - the peer of `create_policy()`. The trainer is selected by the **same provider name** as the inference policy, so the same string that runs a policy also trains one. `TrainSpec` describes the run; `validate()` is a pure preflight that launches nothing if the spec is wrong. Here ACT trains from scratch for two steps on CPU.

> Swap the provider to `"groot"` or `"cosmos3"` and only the provider string changes - the LeRobot, GR00T, and Cosmos training pipelines all sit behind the same `TrainSpec` + `Trainer` lifecycle.

In [ ]:
trainer = create_trainer("lerobot_local", device="cpu")
spec = TrainSpec(
    dataset_root=ROOT,
    base_model="",            # ACT from scratch (smallest CPU path)
    output_dir=OUT,
    steps=2,
    save_freq=2,
    global_batch_size=2,
    extra={"policy_type": "act", "num_workers": 0},
)

problems = trainer.validate(spec)
assert not problems, problems

result = trainer.train(spec)
print("train status:", result.status)
print("checkpoint:", result.checkpoint_dir)

## 3. Load the trained checkpoint back

The exported checkpoint loads through the same `create_policy()` entry point from [notebook 1](./01_getting_started.ipynb) - the loop closes: the thing you trained is a policy you can run.

In [ ]:
policy = create_policy(result.checkpoint_dir)
print("loaded:", type(policy).__name__)
print("record -> train -> export -> load closed on CPU")

## Where to go from here

- Raise `steps`, point `base_model` at a pretrained checkpoint, and run on a GPU for a real policy.
- `stop_recording(bucket=...)` plus `stream_dataset()` let you train straight from a Hugging Face Storage Bucket without a full download.
- The [`07_post_tune_any_policy.py`](../07_post_tune_any_policy.py) script is the same loop in one file.